In [9]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.data_postprocessing import obtain_shoreline
from src.data_processing.dataset_loader import CoastData
from scipy.spatial import cKDTree


In [24]:
def compute_distance(coords_pred, coords_gt):
    # Distance from predicted to GT
    # Create a KDTree for the Ground Truth coordinates
    tree_gt = cKDTree(coords_gt)

    # Find the nearest neighbors in the Ground Truth for each coordinate in the predicted mask
    dists_pred_to_gt, _ = tree_gt.query(coords_pred)

    # Distance from GT to predicted
    # Create a KDTree for the predicted coordinates
    tree_pred = cKDTree(coords_pred)
    # Find the nearest neighbors in the predicted for each coordinate in the GT mask
    dists_gt_to_pred, _ = tree_pred.query(coords_gt)

    return dists_pred_to_gt, dists_gt_to_pred

In [44]:
def calculate_dataset(data_path):
    data = CoastData(data_path)

    filtered_data = data.get_images_and_masks(metadata=True) # All the data

    global_distance_all_points = {
        "dist_pred_to_gt": [],
        "dist_gt_to_pred": [],
        "pred_points": 0,
        "gt_points": 0
    }

    for item in filtered_data:
        coords_gt_u = item['metadata']["image"]["shoreline"]["coordinates"]['u']
        coords_gt_v = item['metadata']["image"]["shoreline"]["coordinates"]['v']
        coords_gt = np.column_stack((coords_gt_u, coords_gt_v))

        coords_pred_u = item['metadata']['image']['predicted_shoreline']["coordinates"]['u']
        coords_pred_v = item['metadata']['image']['predicted_shoreline']["coordinates"]['v']
        coords_pred = np.column_stack((coords_pred_u, coords_pred_v))

        dists_pred_to_gt, dists_gt_to_pred = compute_distance(coords_pred, coords_gt)

        global_distance_all_points["dist_pred_to_gt"].extend(dists_pred_to_gt)
        global_distance_all_points["dist_gt_to_pred"].extend(dists_gt_to_pred)
        global_distance_all_points["pred_points"] += len(coords_pred)
        global_distance_all_points["gt_points"] += len(coords_gt)

    mean_dist_pred_to_gt = np.mean(global_distance_all_points["dist_pred_to_gt"])
    std_dist_pred_to_gt = np.std(global_distance_all_points["dist_pred_to_gt"])
    rmsd_dist_pred_to_gt = np.sqrt(np.mean(np.square(global_distance_all_points["dist_pred_to_gt"])))
    mean_dist_gt_to_pred = np.mean(global_distance_all_points["dist_gt_to_pred"])
    std_dist_gt_to_pred = np.std(global_distance_all_points["dist_gt_to_pred"])
    rmsd_dist_gt_to_pred = np.sqrt(np.mean(np.square(global_distance_all_points["dist_gt_to_pred"])))
    q3_dist_pred_to_gt = np.percentile(global_distance_all_points["dist_pred_to_gt"], 75)
    q3_dist_gt_to_pred = np.percentile(global_distance_all_points["dist_gt_to_pred"], 75)

    ratio = global_distance_all_points["pred_points"] / global_distance_all_points["gt_points"]

    print(f"Number of points pred: {len(global_distance_all_points['dist_pred_to_gt'])}, Number of points GT: {len(global_distance_all_points['dist_gt_to_pred'])}, Ratio: {ratio:.4f}")
    print(f"Mean Absolute Distance (pred->Gt) global: {np.mean(mean_dist_pred_to_gt):.4f} ({std_dist_pred_to_gt:.4f})")
    print(f"Mean Absolute Distance (Gt->pred) global: {np.mean(mean_dist_gt_to_pred):.4f} ({std_dist_gt_to_pred:.4f})")
    print(f"RMSD (pred->Gt) global: {np.mean(rmsd_dist_pred_to_gt):.4f}")
    print(f"RMSD (Gt->pred) global: {np.mean(rmsd_dist_gt_to_pred):.4f}")
    print(f"Q3 75th Percentile (pred->Gt) global: {np.mean(q3_dist_pred_to_gt):.4f}")
    print(f"Q3 75th Percentile (Gt->pred) global: {np.mean(q3_dist_gt_to_pred):.4f}")



In [47]:
data_path_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/UNet/"))
data_path_attention_unet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/AttentionUNet/"))
data_path_deeplabv3 = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/DeepLabV3/"))
data_path_ducknet = os.path.abspath(os.path.join(os.getcwd(), "../../outputs/experiment2/rectified/DuckNet/"))

print("UNet results:")
calculate_dataset(data_path_unet)
print("\nAttention UNet results:")
calculate_dataset(data_path_attention_unet)
print("\nDeepLabV3 results:")
calculate_dataset(data_path_deeplabv3)
print("\nDuckNet results:")
calculate_dataset(data_path_ducknet)

UNet results:
CoastData: global - 174 images
Number of points pred: 102298, Number of points GT: 134174, Ratio: 0.7624
Mean Absolute Distance (pred->Gt) global: 12.0572 (18.5680)
Mean Absolute Distance (Gt->pred) global: 9.7490 (20.7545)
RMSD (pred->Gt) global: 22.1393
RMSD (Gt->pred) global: 22.9301
Q3 75th Percentile (pred->Gt) global: 13.6015
Q3 75th Percentile (Gt->pred) global: 10.1980

Attention UNet results:
CoastData: global - 174 images
Number of points pred: 103294, Number of points GT: 134174, Ratio: 0.7699
Mean Absolute Distance (pred->Gt) global: 13.0488 (21.8696)
Mean Absolute Distance (Gt->pred) global: 11.1721 (26.4124)
RMSD (pred->Gt) global: 25.4667
RMSD (Gt->pred) global: 28.6781
Q3 75th Percentile (pred->Gt) global: 13.3417
Q3 75th Percentile (Gt->pred) global: 10.1980

DeepLabV3 results:
CoastData: global - 174 images
Number of points pred: 97018, Number of points GT: 134174, Ratio: 0.7231
Mean Absolute Distance (pred->Gt) global: 9.9600 (13.9254)
Mean Absolute Dis